<div style="border: 5px solid #003366; border-radius: 8px; padding: 20px;">

### **Course:** DSC550 — Data Mining
### **Name:** Tim Hollis
### **Assignment:** Exercise 3.2
### **Date:** 03/31/2026

---

Reference: Kaggle. (n.d.). *Word2Vec NLP tutorial: Labeled training data* [Dataset]. Retrieved from https://www.kaggle.com/competitions/word2vec-nlp-tutorial/data

</div>

### **Initial Setup:**

In [57]:
# Load Imports
import pandas as pd
import numpy as np
import nltk
import re
import warnings
from IPython.display import display, HTML
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

display(HTML('''
<style>
    div.output_subarea { page-break-inside: avoid; }
    div.jp-MarkdownOutput { page-break-inside: avoid; }
    div.cell { page-break-inside: avoid; }
</style>
'''))

# Suppress warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.precision', 4)

# Load dataset
df = pd.read_csv("labeledTrainData.tsv", sep="\t")

# Verification
print("Setup complete.")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nSentiment counts:")
print(df['sentiment'].map({1: 'Positive', 0: 'Negative'}).value_counts())

Setup complete.
Shape: (25000, 3)

First 5 rows:


,id,sentiment,review
0,5814_8,1,"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just wa..."
1,2381_9,1,"\The Classic War of the Worlds\"" by Timothy Hines is a very entertaining film that obviously goes to great effort and lengths to faithfully recreate H. G. Wells' classic book. Mr. Hines succeeds i..."
2,7759_3,0,"The film starts with a manager (Nicholas Bell) giving welcome investors (Robert Carradine) to Primal Park . A secret project mutating a primal animal using fossilized DNA, like ¨Jurassik Park¨, an..."
3,3630_4,0,"It must be assumed that those who praised this film (\the greatest filmed opera ever,\"" didn't I read somewhere?) either don't care for opera, don't care for Wagner, or don't care about anything e..."
4,9495_8,1,"Superbly trashy and wondrously unpretentious 80's exploitation, hooray! The pre-credits opening sequences somewhat give the false impression that we're dealing with a serious and harrowing drama, ..."



Sentiment counts:
sentiment
Positive    12500
Negative    12500
Name: count, dtype: int64


# **Part 1 Using the TextBlob (and VADER) Sentiment Analyzer**

---

## Step 1: TextBlob Sentiment Classification

We apply TextBlob to compute polarity scores for each review and classify them as positive or negative. A polarity score greater than or equal to zero is considered positive, and a score below zero is considered negative.

---

In [58]:
# Compute polarity for each review
df['polarity'] = df['review'].apply(lambda x: TextBlob(x).sentiment.polarity)

# Convert polarity to predicted sentiment (less than 0 gets 0, all others
# gets 1)
df['tb_pred'] = df['polarity'].apply(lambda p: 1 if p >= 0 else 0)

print("TextBlob classification complete.")
df[['sentiment', 'polarity', 'tb_pred']].head()

TextBlob classification complete.


,sentiment,polarity,tb_pred
0,1,0.0013,1
1,1,0.2563,1
2,0,-0.0539,0
3,0,0.1348,1
4,1,-0.0248,0


---

## Step 2: Evaluate TextBlob Accuracy

We compare TextBlob’s predicted sentiment labels to the true sentiment labels and compute overall accuracy to determine whether the model performs better than random guessing.

---

In [59]:
# Evaluate accuracy
tb_accuracy = accuracy_score(df['sentiment'], df['tb_pred'])

print("TextBlob Accuracy:", round(tb_accuracy, 4))
print("Random Guessing Baseline: 0.50")

TextBlob Accuracy: 0.6852
Random Guessing Baseline: 0.50


---

## Step 3: VADER Sentiment Classification

We apply the VADER sentiment analyzer to compute compound scores for each review and classify them as positive or negative. A compound score greater than or equal to zero is considered positive, and a score below zero is considered negative.

---

In [60]:
analyzer = SentimentIntensityAnalyzer()

# Compute compound score for each review
df['vader_compound'] = df['review'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'])

# Convert compound score to predicted sentiment
df['vader_pred'] = df['vader_compound'].apply(lambda c: 1 if c >= 0 else 0)

# Evaluate accuracy
vader_accuracy = accuracy_score(df['sentiment'], df['vader_pred'])

print("VADER Accuracy:", round(vader_accuracy, 4))
print("Random Guessing Baseline: 0.50")

VADER Accuracy: 0.694
Random Guessing Baseline: 0.50


<div style="border: 3px solid #003366; border-radius: 8px; padding: 20px;">

## Summary

TextBlob and VADER were applied to classify IMDB movie reviews as positive or negative using polarity and compound scores. TextBlob achieved an accuracy of **0.6852**, while VADER achieved **0.6940**, with both models performing better than the 0.50 random‑guessing baseline. These results show that simple, rule‑based sentiment analyzers can capture broad sentiment patterns in movie reviews, though performance varies slightly between methods.

</div>

# **Part 2: Prepping Text for a Custom Model**

We now prepare the IMDB reviews for machine learning by applying standard text‑cleaning steps. This includes lowercasing, removing punctuation, removing stop words, and stemming. 

---

## Step 1: Lowercasing

---

In [61]:
# Lowercasing
df['clean_review'] = df['review'].str.lower()

print("Lowercasing complete.")
df['clean_review'].head()

Lowercasing complete.


0    with all this stuff going down at the moment with mj i've started listening to his music, watching the odd documentary here and there, watched the wiz and watched moonwalker again. maybe i just wa...
1    \the classic war of the worlds\" by timothy hines is a very entertaining film that obviously goes to great effort and lengths to faithfully recreate h. g. wells' classic book. mr. hines succeeds i...
2    the film starts with a manager (nicholas bell) giving welcome investors (robert carradine) to primal park . a secret project mutating a primal animal using fossilized dna, like ¨jurassik park¨, an...
3    it must be assumed that those who praised this film (\the greatest filmed opera ever,\" didn't i read somewhere?) either don't care for opera, don't care for wagner, or don't care about anything e...
4    superbly trashy and wondrously unpretentious 80's exploitation, hooray! the pre-credits opening sequences somewhat give the false impression that we're dealing with a serious 

---

## Step 2: Remove Punctuation

---

In [62]:
# Remove punctuation
df['clean_review'] = df['clean_review'].apply(
    lambda x: re.sub(r'[^\w\s]', '', x))

print("Punctuation removal complete.")
df['clean_review'].head()

Punctuation removal complete.


0    with all this stuff going down at the moment with mj ive started listening to his music watching the odd documentary here and there watched the wiz and watched moonwalker again maybe i just want t...
1    the classic war of the worlds by timothy hines is a very entertaining film that obviously goes to great effort and lengths to faithfully recreate h g wells classic book mr hines succeeds in doing ...
2    the film starts with a manager nicholas bell giving welcome investors robert carradine to primal park  a secret project mutating a primal animal using fossilized dna like jurassik park and some sc...
3    it must be assumed that those who praised this film the greatest filmed opera ever didnt i read somewhere either dont care for opera dont care for wagner or dont care about anything except their d...
4    superbly trashy and wondrously unpretentious 80s exploitation hooray the precredits opening sequences somewhat give the false impression that were dealing with a serious and h

<div style="page-break-after: always;"></div>

---

## Step 3: Remove Stop Words

---

In [63]:
# Remove stop words
stop_words = set(stopwords.words('english'))

df['clean_review'] = df['clean_review'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)

print("Stop word removal complete.")
df['clean_review'].head()

Stop word removal complete.


0    stuff going moment mj ive started listening music watching odd documentary watched wiz watched moonwalker maybe want get certain insight guy thought really cool eighties maybe make mind whether gu...
1    classic war worlds timothy hines entertaining film obviously goes great effort lengths faithfully recreate h g wells classic book mr hines succeeds watched film appreciated fact standard predictab...
2    film starts manager nicholas bell giving welcome investors robert carradine primal park secret project mutating primal animal using fossilized dna like jurassik park scientists resurrect one natur...
3    must assumed praised film greatest filmed opera ever didnt read somewhere either dont care opera dont care wagner dont care anything except desire appear cultured either representation wagners swa...
4    superbly trashy wondrously unpretentious 80s exploitation hooray precredits opening sequences somewhat give false impression dealing serious harrowing drama need fear barely t

---

## Step 4: Stemming

We apply the Porter Stemmer to reduce words to their root forms. 

---

In [64]:
# Stemming
ps = PorterStemmer()

df['clean_review'] = df['clean_review'].apply(
    lambda x: ' '.join([ps.stem(word) for word in x.split()])
)

print("Stemming complete.")
df['clean_review'].head()

Stemming complete.


0    stuff go moment mj ive start listen music watch odd documentari watch wiz watch moonwalk mayb want get certain insight guy thought realli cool eighti mayb make mind whether guilti innoc moonwalk p...
1    classic war world timothi hine entertain film obvious goe great effort length faith recreat h g well classic book mr hine succe watch film appreci fact standard predict hollywood fare come everi y...
2    film start manag nichola bell give welcom investor robert carradin primal park secret project mutat primal anim use fossil dna like jurassik park scientist resurrect one natur fearsom predat sabre...
3    must assum prais film greatest film opera ever didnt read somewher either dont care opera dont care wagner dont care anyth except desir appear cultur either represent wagner swansong movi strike u...
4    superbl trashi wondrous unpretenti 80 exploit hooray precredit open sequenc somewhat give fals impress deal seriou harrow drama need fear bare ten minut later neck nonsens cha

---

## Step 5: Bag-of-Words Matrix

We create a bag-of-words (BoW) matrix from the fully preprocessed and stemmed text. Each row represents a movie review, and each column represents a unique word. The resulting sparse matrix contains word-count vectors suitable for model building.

---

In [65]:
# BoW matrix
vectorizer_bow = CountVectorizer()
bow_matrix = vectorizer_bow.fit_transform(df['clean_review'])

print("Bag-of-Words matrix created.")
print("Dimensions:", bow_matrix.shape)

Bag-of-Words matrix created.
Dimensions: (25000, 92532)


---

## Step 6: TF‑IDF Matrix

We now create a TF‑IDF matrix from the same stemmed text. TF‑IDF scales word counts by how informative each term is across the corpus. The resulting matrix has the same dimensions as the Bag‑of‑Words matrix.

---

In [66]:
# TF-IDF Matrix
vectorizer_tfidf = TfidfVectorizer()
tfidf_matrix = vectorizer_tfidf.fit_transform(df['clean_review'])

print("TF-IDF matrix created.")
print("Dimensions:", tfidf_matrix.shape)

TF-IDF matrix created.
Dimensions: (25000, 92532)


---

## Bag-of-Words vs. TF-IDF Comparison

| Feature                     | Bag-of-Words (BoW)                                   | TF-IDF                                              |
|-----------------------------|-------------------------------------------------------|------------------------------------------------------|
| What it measures            | Raw word counts                                       | Weighted importance of words across documents        |
| Matrix type                 | Sparse count matrix                                   | Sparse floating‑point matrix                         |
| Values                      | Integer counts (0, 1, 2, …)                           | Continuous weights (0.0–1.0+)                        |
| Effect of common words      | Very common words dominate                            | Common words are down‑weighted                       |
| Effect of rare words        | Rare words have low impact                            | Rare but informative words get higher weight         |
| Vocabulary size             | Same as TF‑IDF                                        | Same as BoW                                          |
| Dimensions (rows × columns) | `(25000, 92532)`                                      | `(25000, 92532)`                                     |


<div style="border: 3px solid black; border-radius: 8px; padding: 20px;">

## Summary

In this section, we completed a full text‑preprocessing pipeline and generated two feature‑extraction matrices used for downstream machine learning tasks. The preprocessing steps included converting all text to lowercase, removing punctuation and special characters, removing stop words, and applying NLTK’s PorterStemmer to reduce words to their root forms. These transformations standardize the text and reduce vocabulary size, improving the quality of the resulting feature representations.

After preprocessing, we created two matrices from the stemmed text:

1. **Bag-of-Words (BoW) Matrix**  
   - Constructed using `CountVectorizer`  
   - Represents each review as a vector of raw word counts  
   - Dimensions: `(25000, 92532)`  
  
2. **TF-IDF Matrix**  
   - Constructed using `TfidfVectorizer`  
   - Represents each review using term frequency–inverse document frequency weights  
   - Dimensions: `(25000, 92532)`  

</div>

<div style="border: 5px solid #228B22; border-radius: 8px; padding: 20px;">

## Reflection 

### Overall Assessment
This assignment was a solid introduction to practical sentiment analysis and the foundational text‑processing steps required for building custom NLP models. Working with the IMDB movie review dataset helped connect theoretical concepts to hands‑on implementation, and it highlighted how raw text must be transformed before any meaningful modeling can occur. The workflow—from using prebuilt analyzers to constructing bag‑of‑words and TF‑IDF matrices—reinforced the importance of both interpretability and careful preprocessing in natural language tasks.

### Straightforward Portions
- Loading the dataset and verifying the distribution of positive vs. negative labels  
- Applying TextBlob polarity scoring and classifying reviews based on threshold rules  
- Using VADER for comparison, since it provides sentiment scores directly  
- Implementing basic preprocessing steps such as lowercasing, punctuation removal, and stop‑word filtering  
- Generating bag‑of‑words and TF‑IDF matrices using scikit‑learn’s vectorizers  

### More Challenging Portions
- Interpreting the limitations of rule‑based sentiment analyzers, especially with nuanced or mixed‑tone reviews  
- Ensuring preprocessing steps were applied in the correct order without unintentionally removing meaningful tokens  
- Managing the size and sparsity of the resulting matrices and understanding their structure  
- Recognizing how stemming affects vocabulary size and the interpretability of the final feature space   
